# Inspect One Compact Byte Batch - v4

Load one precomputed v4 compact event-chunk batch, inspect tensor shapes and byte values, optionally restore a checkpoint, render a Keras-style model summary/graph, run one masked-autoencoder inference, and calculate one reconstruction loss.

In [ ]:
from pathlib import Path

MODEL_VERSION = "v4"
WORKSPACE = None  # Leave None to auto-detect local repo or workstation runtime root.
PRECOMPUTED_CHUNK_ROOT = Path(r"D:\market-data\prepared\us_stocks_sip\v4_compact_event_chunks_v1")
REFERENCE_DIR = None  # Leave None to use research/market_references/massive inside the resolved workspace.
SESSION_DATE = "2025-11-03"
TICKERS = ("ALL",)
BATCH_SIZE = 8
EVENTS_PER_CHUNK = 128
DEVICE = "cuda"
USE_AMP = False

# Paste a checkpoint path, or leave empty to auto-load the newest checkpoint under the v4 runtime tree.
CHECKPOINT_PATH = ""
RUNTIME_ROOT = Path(r"D:\TradingML\runtimes\masked_event_model\v4\pretrain")
ARCHITECTURE_OUTPUT_DIR = Path(r"D:\TradingML\runtimes\masked_event_model\v4\notebook_artifacts\inspect_architecture")


In [ ]:
import json
import math
import sys
from pathlib import Path

import numpy as np
import polars as pl
import torch
from IPython.display import Image, Markdown, display
from torch.utils.data import DataLoader


def resolve_workspace(explicit):
    if explicit:
        path = Path(explicit)
        if (path / "research" / "masked_event_model" / MODEL_VERSION).exists():
            return path
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates.extend([
        Path(r"D:\TradingCodes\quant-research-workbench"),
        Path(r"D:\TradingML\codes\masked_event_model\v4"),
        Path(r"\\DESKTOP-SAAI85T\Workstation-D\TradingML\codes\masked_event_model\v4"),
    ])
    for path in candidates:
        if (path / "research" / "masked_event_model" / MODEL_VERSION).exists():
            return path
    raise FileNotFoundError("Could not find a workspace root containing research/masked_event_model/v4.")


WORKSPACE = resolve_workspace(WORKSPACE)
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from research.masked_event_model.v4.config import DataConfig, ExperimentConfig, LossConfig, MaskConfig, ModelConfig, TrainConfig
from research.masked_event_model.v4.losses import masked_byte_bce_loss, pack_bits, unpack_bits
from research.masked_event_model.v4.masking import build_byte_masks
from research.masked_event_model.v4.model import CompactByteMaskedAutoencoder
from research.mlops.compact_events import HEADER_BYTES, EVENT_BYTES, PrecomputedChunkDataConfig, PrecomputedV4ChunkIterableDataset, resolve_precomputed_chunk_root

np.set_printoptions(precision=5, suppress=True)
torch.set_printoptions(precision=5, sci_mode=False)
torch.set_float32_matmul_precision("high")

device = torch.device(DEVICE if DEVICE == "cpu" or torch.cuda.is_available() else "cpu")
reference_dir = Path(REFERENCE_DIR) if REFERENCE_DIR else WORKSPACE / "research" / "market_references" / "massive"
chunk_root = resolve_precomputed_chunk_root(PRECOMPUTED_CHUNK_ROOT)
print("workspace:", WORKSPACE)
print("chunk_root:", chunk_root)
print("device:", device)


In [ ]:
dataset = PrecomputedV4ChunkIterableDataset(
    PrecomputedChunkDataConfig(
        chunk_root=PRECOMPUTED_CHUNK_ROOT,
        start_date=SESSION_DATE,
        end_date=SESSION_DATE,
        tickers=TICKERS,
        batch_size=BATCH_SIZE,
        events_per_chunk=EVENTS_PER_CHUNK,
        seed=17,
        shard_cache_size=2,
    )
)
loader = DataLoader(dataset, batch_size=None, num_workers=0)
batch = next(iter(loader))
print("shard count:", len(dataset.shards))
print("batch keys:", sorted(batch.keys()))
for key, value in batch.items():
    if torch.is_tensor(value):
        print(f"{key:24s} shape={tuple(value.shape)} dtype={value.dtype} min={int(value.min()) if value.numel() else 'n/a'} max={int(value.max()) if value.numel() else 'n/a'}")
    else:
        preview = value[:5] if isinstance(value, list) else value
        print(f"{key:24s} type={type(value).__name__} preview={preview}")


In [ ]:
row = 0
header = batch["header_uint8"][row].numpy()
events = batch["events_uint8"][row].numpy()
print("ticker:", batch["ticker"][row])
print("origin_timestamp_ns:", int(batch["origin_timestamp_ns"][row]))
print("session_date:", batch["session_date"][row])
display(pl.DataFrame({"header_byte_index": list(range(len(header))), "uint8": header.tolist()}))
display(pl.DataFrame(events[:20], schema=[f"byte_{idx:02d}" for idx in range(events.shape[1])], orient="row"))


In [ ]:
def latest_checkpoint(runtime_root):
    candidates = list(runtime_root.glob("*/checkpoints/latest.pt")) + list(runtime_root.glob("*/checkpoints/checkpoint_latest.pt")) + list(runtime_root.glob("*/checkpoints/*.pt"))
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_mtime)

checkpoint_path = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else latest_checkpoint(RUNTIME_ROOT)
checkpoint = None
checkpoint_config = None
if checkpoint_path and checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    checkpoint_config = checkpoint.get("config")
    print("loaded checkpoint:", checkpoint_path)
    print("checkpoint step:", checkpoint.get("step"))
else:
    print("No checkpoint loaded; using fresh randomly initialized model.")

model_config = ModelConfig(**{k: v for k, v in (checkpoint_config or {}).get("model", {}).items() if k in ModelConfig.__dataclass_fields__}) if checkpoint_config else ModelConfig()
mask_config = MaskConfig(**{k: v for k, v in (checkpoint_config or {}).get("masks", {}).items() if k in MaskConfig.__dataclass_fields__}) if checkpoint_config else MaskConfig()
loss_config = LossConfig(**{k: v for k, v in (checkpoint_config or {}).get("losses", {}).items() if k in LossConfig.__dataclass_fields__}) if checkpoint_config else LossConfig()
model = CompactByteMaskedAutoencoder(events_per_chunk=EVENTS_PER_CHUNK, config=model_config).to(device)
if checkpoint is not None:
    model.load_state_dict(checkpoint.get("model", checkpoint.get("model_state_dict", checkpoint)), strict=True)
model.eval()
print(f"model parameters={sum(p.numel() for p in model.parameters()):,}")
print("model_config:", model_config)


In [ ]:
# Keras-style model summary and graph view.
# Optional packages for richer output:
#   pip install torchinfo torchview graphviz
# Graph rendering also needs the Graphviz dot executable on PATH.

ARCHITECTURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_header = torch.zeros((1, HEADER_BYTES), dtype=torch.uint8, device=device)
summary_events = torch.zeros((1, EVENTS_PER_CHUNK, EVENT_BYTES), dtype=torch.uint8, device=device)
summary_masks = build_byte_masks(summary_header, summary_events, mask_config)

try:
    from torchinfo import summary
    model_summary = summary(
        model,
        input_data=(summary_header, summary_events, summary_masks),
        depth=8,
        col_names=("input_size", "output_size", "num_params", "trainable"),
        verbose=0,
    )
    summary_text = str(model_summary)
except Exception as exc:
    summary_text = f"torchinfo summary failed: {exc}\n\n{model}"

summary_path = ARCHITECTURE_OUTPUT_DIR / f"masked_event_model_{MODEL_VERSION}_summary.txt"
summary_path.write_text(summary_text, encoding="utf-8")
display(Markdown("```text\n" + summary_text[:50000] + ("\n... truncated ..." if len(summary_text) > 50000 else "") + "\n```"))
print("summary_path:", summary_path)

try:
    from torchview import draw_graph
    graph = draw_graph(model, input_data=(summary_header, summary_events, summary_masks), expand_nested=True, depth=3, save_graph=True, directory=str(ARCHITECTURE_OUTPUT_DIR), filename=f"masked_event_model_{MODEL_VERSION}_graph")
    png_path = ARCHITECTURE_OUTPUT_DIR / f"masked_event_model_{MODEL_VERSION}_graph.png"
    print("graph_png:", png_path)
    if png_path.exists():
        display(Image(filename=str(png_path)))
except Exception as exc:
    print("torchview graph failed:", repr(exc))


In [ ]:
def move_tensor_batch(batch_dict, device):
    return {key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value for key, value in batch_dict.items()}

model.eval()
device_batch = move_tensor_batch(batch, device)
masks = build_byte_masks(device_batch["header_uint8"], device_batch["events_uint8"], mask_config)
with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP and device.type == "cuda"):
    output = model(device_batch["header_uint8"], device_batch["events_uint8"], masks)
    result = masked_byte_bce_loss(output, device_batch["header_uint8"], device_batch["events_uint8"], masks, loss_config, include_diagnostics=True)
    chunk_embedding = model.encode(device_batch["header_uint8"], device_batch["events_uint8"])
    event_embeddings = model.encode_events(device_batch["header_uint8"], device_batch["events_uint8"])

print("loss:", float(result.loss.detach().cpu()))
print("metrics:", json.dumps(result.metrics, indent=2, sort_keys=True))
print("header logits:", tuple(output.header_bit_logits.shape))
print("event logits:", tuple(output.event_bit_logits.shape))
print("chunk_embedding:", tuple(chunk_embedding.shape))
print("event_embeddings:", tuple(event_embeddings.shape))


In [ ]:
header_indices = output.header_indices.detach().cpu()
event_indices = output.event_indices.detach().cpu()
header_probs = torch.sigmoid(output.header_bit_logits.detach().cpu())
event_probs = torch.sigmoid(output.event_bit_logits.detach().cpu())
header_pred = pack_bits(header_probs >= 0.5).cpu() if header_probs.numel() else torch.empty(0, dtype=torch.long)
event_pred = pack_bits(event_probs >= 0.5).cpu() if event_probs.numel() else torch.empty(0, dtype=torch.long)

rows = []
for idx, pred in zip(header_indices[:50], header_pred[:50]):
    b, byte_idx = [int(x) for x in idx]
    target = int(batch["header_uint8"][b, byte_idx])
    rows.append({"group": "header", "sample": b, "event": None, "byte": byte_idx, "target_uint8": target, "pred_uint8": int(pred), "abs_error": abs(int(pred) - target)})
for idx, pred in zip(event_indices[:200], event_pred[:200]):
    b, event_idx, byte_idx = [int(x) for x in idx]
    target = int(batch["events_uint8"][b, event_idx, byte_idx])
    rows.append({"group": "event", "sample": b, "event": event_idx, "byte": byte_idx, "target_uint8": target, "pred_uint8": int(pred), "abs_error": abs(int(pred) - target)})
pl.DataFrame(rows)
